# Advanced Causal Estimators

This notebook demonstrates advanced causal inference methods available in the datascienceutils library.

## Methods Covered:
- **Regression Discontinuity Design (RDD)** - Sharp cutoff analysis
- **Synthetic Control Method** - Comparative case studies
- **Mediation Analysis** - Direct and indirect effects
- **Conditional Average Treatment Effect (CATE)** - Heterogeneous treatment effects

## Prerequisites
```bash
maturin develop --release
pip install jupyter matplotlib numpy
```

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import datascienceutils as dsu

np.random.seed(42)
print(f"Using datascienceutils v{dsu.__version__}")

Using datascienceutils v0.1.0


## 1. Regression Discontinuity Design (RDD)

RDD estimates treatment effects when assignment is determined by a threshold.

**Scenario**: Estimating the effect of scholarship eligibility (test score ≥ 70) on college graduation.

In [2]:
# Generate synthetic RDD data
n = 500
cutoff = 70.0

# Running variable: test score
running_var = np.random.uniform(40, 100, n)

# Treatment: scholarship (1 if score >= cutoff)
treatment_rdd = (running_var >= cutoff).astype(float)

# True treatment effect = 15 percentage points
true_effect_rdd = 15.0

# Outcome: graduation rate (smooth function of test score + jump at cutoff)
# Base graduation rate increases with test score
outcome_rdd = (
    30 +  # Baseline
    0.5 * running_var +  # Smooth relationship with test score
    true_effect_rdd * treatment_rdd +  # Treatment effect (discontinuity)
    np.random.randn(n) * 5  # Noise
)

print(f"Sample size: {n}")
print(f"Cutoff: {cutoff}")
print(f"Below cutoff: {(running_var < cutoff).sum()}")
print(f"Above cutoff: {(running_var >= cutoff).sum()}")
print(f"\nTrue treatment effect: {true_effect_rdd:.2f} percentage points")

Sample size: 500
Cutoff: 70.0
Below cutoff: 241
Above cutoff: 259

True treatment effect: 15.00 percentage points


In [3]:
# Estimate RDD effect
rdd_estimate = dsu.regression_discontinuity(running_var, outcome_rdd, cutoff)

print(f"RDD Estimate: {rdd_estimate:.2f} percentage points")
print(f"True effect: {true_effect_rdd:.2f} percentage points")
print(f"Estimation error: {abs(rdd_estimate - true_effect_rdd):.2f}")

AttributeError: module 'datascienceutils' has no attribute 'regression_discontinuity'

In [ ]:
# Visualize RDD
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot with discontinuity
below = running_var < cutoff
above = running_var >= cutoff

axes[0].scatter(running_var[below], outcome_rdd[below], alpha=0.5, s=30, 
                label='No Scholarship', color='blue')
axes[0].scatter(running_var[above], outcome_rdd[above], alpha=0.5, s=30, 
                label='Scholarship', color='red')

# Fit lines on both sides
x_below = running_var[below]
y_below = outcome_rdd[below]
z_below = np.polyfit(x_below, y_below, 1)
p_below = np.poly1d(z_below)

x_above = running_var[above]
y_above = outcome_rdd[above]
z_above = np.polyfit(x_above, y_above, 1)
p_above = np.poly1d(z_above)

x_plot_below = np.linspace(x_below.min(), cutoff, 100)
x_plot_above = np.linspace(cutoff, x_above.max(), 100)

axes[0].plot(x_plot_below, p_below(x_plot_below), 'b-', linewidth=2, label='Fit (Below)')
axes[0].plot(x_plot_above, p_above(x_plot_above), 'r-', linewidth=2, label='Fit (Above)')

# Mark the discontinuity
axes[0].axvline(x=cutoff, color='green', linestyle='--', linewidth=2, label='Cutoff')

# Annotate the jump
y_at_cutoff_below = p_below(cutoff)
y_at_cutoff_above = p_above(cutoff)
axes[0].annotate('', xy=(cutoff + 1, y_at_cutoff_below), xytext=(cutoff + 1, y_at_cutoff_above),
                arrowprops=dict(arrowstyle='<->', color='green', lw=2))
axes[0].text(cutoff + 2, (y_at_cutoff_below + y_at_cutoff_above)/2, 
            f'Jump\n{rdd_estimate:.1f}', fontsize=10, color='green', fontweight='bold')

axes[0].set_xlabel('Test Score (Running Variable)')
axes[0].set_ylabel('Graduation Rate (%)')
axes[0].set_title('Regression Discontinuity Design')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Histogram of running variable
axes[1].hist(running_var, bins=30, edgecolor='black', alpha=0.7)
axes[1].axvline(x=cutoff, color='green', linestyle='--', linewidth=2, label='Cutoff')
axes[1].set_xlabel('Test Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Running Variable')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Synthetic Control Method

Synthetic control creates a weighted combination of control units to match the treated unit.

**Scenario**: Estimating the effect of a new tax policy in one state using other states as controls.

In [ ]:
# Generate synthetic control data
n_pre = 10   # Pre-treatment periods
n_post = 5   # Post-treatment periods
n_controls = 5  # Number of control units

# Pre-treatment outcomes for treated unit
treated_pre = np.array([100, 102, 105, 103, 107, 110, 108, 112, 115, 113])

# Pre-treatment outcomes for control units (similar trends)
control_pre = np.array([
    [98, 100, 103, 101, 105, 108, 106, 110, 113, 111],   # Control 1
    [102, 104, 107, 105, 109, 112, 110, 114, 117, 115],  # Control 2
    [95, 97, 100, 98, 102, 105, 103, 107, 110, 108],     # Control 3
    [105, 107, 110, 108, 112, 115, 113, 117, 120, 118],  # Control 4
    [100, 102, 105, 103, 107, 110, 108, 112, 115, 113],  # Control 5 (similar to treated)
])

# True treatment effect = -8 (negative impact)
true_effect_sc = -8.0

# Post-treatment outcomes (controls continue trend, treated has effect)
treated_post = np.array([117, 110, 112, 115, 113])  # Drops due to policy
control_post = np.array([
    [115, 117, 119, 121, 123],  # Control 1 continues trend
    [119, 121, 123, 125, 127],  # Control 2
    [112, 114, 116, 118, 120],  # Control 3
    [122, 124, 126, 128, 130],  # Control 4
    [117, 119, 121, 123, 125],  # Control 5
])

print(f"Pre-treatment periods: {n_pre}")
print(f"Post-treatment periods: {n_post}")
print(f"Number of control units: {n_controls}")
print(f"\nTreated unit pre-treatment mean: {treated_pre.mean():.2f}")
print(f"Treated unit post-treatment mean: {treated_post.mean():.2f}")

In [ ]:
# Estimate synthetic control effect
sc_estimate = dsu.synthetic_control(treated_pre, treated_post, control_pre, control_post)

print(f"Synthetic Control Estimate: {sc_estimate:.2f}")
print(f"True effect: {true_effect_sc:.2f}")
print(f"Estimation error: {abs(sc_estimate - true_effect_sc):.2f}")

In [ ]:
# Visualize synthetic control
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time series plot
time_pre = np.arange(n_pre)
time_post = np.arange(n_pre, n_pre + n_post)
time_all = np.arange(n_pre + n_post)

# Plot treated unit
treated_all = np.concatenate([treated_pre, treated_post])
axes[0].plot(time_all, treated_all, 'o-', linewidth=2, markersize=8, 
             label='Treated Unit', color='red')

# Plot control units (lighter)
for i in range(n_controls):
    control_all = np.concatenate([control_pre[i], control_post[i]])
    axes[0].plot(time_all, control_all, '--', alpha=0.3, linewidth=1, 
                label=f'Control {i+1}' if i < 2 else '', color='gray')

# Mark intervention point
axes[0].axvline(x=n_pre - 0.5, color='green', linestyle='--', linewidth=2, 
                label='Intervention')

# Shade post-treatment period
axes[0].axvspan(n_pre - 0.5, n_pre + n_post - 0.5, alpha=0.1, color='yellow')

axes[0].set_xlabel('Time Period')
axes[0].set_ylabel('Outcome')
axes[0].set_title('Synthetic Control Analysis')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Effect over time
# Calculate what treated would have been without intervention (using control average)
counterfactual_post = control_post.mean(axis=0)
effects_over_time = treated_post - counterfactual_post

axes[1].bar(range(n_post), effects_over_time, alpha=0.7, edgecolor='black', color='red')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1].axhline(y=sc_estimate, color='green', linestyle='--', linewidth=2, 
                label=f'Average Effect: {sc_estimate:.1f}')
axes[1].set_xlabel('Post-Treatment Period')
axes[1].set_ylabel('Treatment Effect')
axes[1].set_title('Effect Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 3. Mediation Analysis

Mediation analysis decomposes total effect into direct and indirect (mediated) effects.

**Scenario**: How does exercise (treatment) affect weight loss (outcome) through metabolism (mediator)?

In [ ]:
# Generate synthetic mediation data
n = 400

# Treatment: exercise program (0 or 1)
treatment_med = np.random.binomial(1, 0.5, n).astype(float)

# Mediator: metabolism rate (affected by exercise)
# Exercise increases metabolism by 5 units on average
mediator = (
    50 +  # Baseline metabolism
    5 * treatment_med +  # Treatment effect on mediator
    np.random.randn(n) * 3  # Noise
)

# Outcome: weight loss (affected by both exercise and metabolism)
# Direct effect of exercise: 3 lbs
# Effect of metabolism: 0.4 lbs per unit
# Indirect effect: 5 * 0.4 = 2 lbs (through metabolism)
# Total effect: 3 + 2 = 5 lbs
outcome_med = (
    10 +  # Baseline weight loss
    3 * treatment_med +  # Direct effect
    0.4 * mediator +  # Mediator effect
    np.random.randn(n) * 2  # Noise
)

true_total_effect = 5.0
true_direct_effect = 3.0
true_indirect_effect = 2.0

print(f"Sample size: {n}")
print(f"Treated: {int(treatment_med.sum())}")
print(f"Control: {int((1-treatment_med).sum())}")
print(f"\nTrue effects:")
print(f"  Total effect: {true_total_effect:.2f} lbs")
print(f"  Direct effect: {true_direct_effect:.2f} lbs")
print(f"  Indirect effect (mediated): {true_indirect_effect:.2f} lbs")

In [ ]:
# Estimate mediation effects
total_eff, direct_eff, indirect_eff = dsu.mediation_analysis(treatment_med, mediator, outcome_med)

print(f"Estimated Total Effect: {total_eff:.2f} lbs")
print(f"Estimated Direct Effect: {direct_eff:.2f} lbs")
print(f"Estimated Indirect Effect: {indirect_eff:.2f} lbs")
print(f"\nProportion mediated: {100 * indirect_eff / total_eff:.1f}%")

In [ ]:
# Visualize mediation
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Treatment → Mediator
bp1 = axes[0].boxplot([mediator[treatment_med == 0], mediator[treatment_med == 1]], 
                       labels=['No Exercise', 'Exercise'], patch_artist=True)
bp1['boxes'][0].set_facecolor('lightblue')
bp1['boxes'][1].set_facecolor('lightcoral')
axes[0].set_ylabel('Metabolism Rate')
axes[0].set_title('Treatment → Mediator')
axes[0].grid(True, alpha=0.3, axis='y')

# Mediator → Outcome
axes[1].scatter(mediator, outcome_med, alpha=0.5, s=30, c=treatment_med, 
                cmap='coolwarm', edgecolors='black', linewidth=0.5)
z = np.polyfit(mediator, outcome_med, 1)
p = np.poly1d(z)
axes[1].plot(mediator, p(mediator), "g--", linewidth=2, label=f'Fit: y={z[0]:.2f}x+{z[1]:.1f}')
axes[1].set_xlabel('Metabolism Rate')
axes[1].set_ylabel('Weight Loss (lbs)')
axes[1].set_title('Mediator → Outcome')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Effect decomposition
labels = ['Total\nEffect', 'Direct\nEffect', 'Indirect\nEffect\n(Mediated)']
effects = [total_eff, direct_eff, indirect_eff]
colors_bar = ['purple', 'blue', 'orange']

bars = axes[2].bar(labels, effects, color=colors_bar, alpha=0.7, edgecolor='black')
axes[2].set_ylabel('Effect Size (lbs)')
axes[2].set_title('Mediation Decomposition')
axes[2].grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, effect in zip(bars, effects):
    height = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width()/2., height,
                f'{effect:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nInterpretation:")
print(f"  Exercise increases weight loss by {total_eff:.2f} lbs total")
print(f"  {direct_eff:.2f} lbs is the direct effect of exercise")
print(f"  {indirect_eff:.2f} lbs is mediated through increased metabolism")

## 4. Conditional Average Treatment Effect (CATE)

CATE estimates how treatment effects vary across different subgroups.

**Scenario**: How does the effect of a drug vary by patient age and health status?

In [ ]:
# Generate synthetic CATE data
n = 500

# Covariates: age and baseline health score
covariates = np.random.randn(n, 2)
covariates[:, 0] = covariates[:, 0] * 15 + 50  # Age: mean=50, std=15
covariates[:, 1] = covariates[:, 1] * 10 + 70  # Health: mean=70, std=10

# Treatment assignment
treatment_cate = np.random.binomial(1, 0.5, n).astype(float)

# Heterogeneous treatment effect:
# - Younger patients benefit more
# - Healthier patients benefit more
# Base effect = 10, decreases with age, increases with health
true_cate = (
    10 +  # Base effect
    -0.2 * (covariates[:, 0] - 50) +  # Age interaction (negative)
    0.1 * (covariates[:, 1] - 70)     # Health interaction (positive)
)

# Outcome: recovery score
outcome_cate = (
    50 +  # Baseline
    0.3 * covariates[:, 0] +  # Age effect
    0.5 * covariates[:, 1] +  # Health effect
    true_cate * treatment_cate +  # Heterogeneous treatment effect
    np.random.randn(n) * 5  # Noise
)

print(f"Sample size: {n}")
print(f"Treated: {int(treatment_cate.sum())}")
print(f"Control: {int((1-treatment_cate).sum())}")
print(f"\nCovariates:")
print(f"  Age: mean={covariates[:, 0].mean():.1f}, range=[{covariates[:, 0].min():.1f}, {covariates[:, 0].max():.1f}]")
print(f"  Health: mean={covariates[:, 1].mean():.1f}, range=[{covariates[:, 1].min():.1f}, {covariates[:, 1].max():.1f}]")

In [ ]:
# Estimate CATE
cate_estimates = dsu.conditional_ate(treatment_cate, outcome_cate, covariates)

print(f"CATE estimates:")
print(f"  Mean: {cate_estimates.mean():.2f}")
print(f"  Std: {cate_estimates.std():.2f}")
print(f"  Min: {cate_estimates.min():.2f}")
print(f"  Max: {cate_estimates.max():.2f}")
print(f"\nTrue CATE:")
print(f"  Mean: {true_cate.mean():.2f}")
print(f"  Std: {true_cate.std():.2f}")

In [ ]:
# Visualize CATE heterogeneity
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# CATE vs Age
axes[0, 0].scatter(covariates[:, 0], cate_estimates, alpha=0.5, s=30, c=cate_estimates, 
                   cmap='RdYlGn', edgecolors='black', linewidth=0.5)
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Estimated CATE')
axes[0, 0].set_title('Treatment Effect by Age')
axes[0, 0].grid(True, alpha=0.3)

# Add trend line
z_age = np.polyfit(covariates[:, 0], cate_estimates, 1)
p_age = np.poly1d(z_age)
axes[0, 0].plot(covariates[:, 0], p_age(covariates[:, 0]), "r--", linewidth=2, 
                label=f'Trend: {z_age[0]:.3f}x+{z_age[1]:.1f}')
axes[0, 0].legend()

# CATE vs Health
axes[0, 1].scatter(covariates[:, 1], cate_estimates, alpha=0.5, s=30, c=cate_estimates, 
                   cmap='RdYlGn', edgecolors='black', linewidth=0.5)
axes[0, 1].set_xlabel('Baseline Health Score')
axes[0, 1].set_ylabel('Estimated CATE')
axes[0, 1].set_title('Treatment Effect by Health')
axes[0, 1].grid(True, alpha=0.3)

# Add trend line
z_health = np.polyfit(covariates[:, 1], cate_estimates, 1)
p_health = np.poly1d(z_health)
axes[0, 1].plot(covariates[:, 1], p_health(covariates[:, 1]), "r--", linewidth=2, 
                label=f'Trend: {z_health[0]:.3f}x+{z_health[1]:.1f}')
axes[0, 1].legend()

# Distribution of CATE
axes[1, 0].hist(cate_estimates, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
axes[1, 0].axvline(cate_estimates.mean(), color='red', linestyle='--', linewidth=2, 
                   label=f'Mean: {cate_estimates.mean():.2f}')
axes[1, 0].set_xlabel('Estimated CATE')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Treatment Effects')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Subgroup analysis
# Divide into age groups
young = covariates[:, 0] < 50
old = covariates[:, 0] >= 50

subgroups = ['Young\n(Age < 50)', 'Old\n(Age ≥ 50)', 'Overall']
subgroup_effects = [
    cate_estimates[young].mean(),
    cate_estimates[old].mean(),
    cate_estimates.mean()
]
colors_sub = ['green', 'orange', 'blue']

bars = axes[1, 1].bar(subgroups, subgroup_effects, color=colors_sub, alpha=0.7, edgecolor='black')
axes[1, 1].set_ylabel('Average Treatment Effect')
axes[1, 1].set_title('Subgroup Analysis')
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, effect in zip(bars, subgroup_effects):
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{effect:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nSubgroup effects:")
print(f"  Young patients (age < 50): {cate_estimates[young].mean():.2f}")
print(f"  Old patients (age ≥ 50): {cate_estimates[old].mean():.2f}")
print(f"  Difference: {cate_estimates[young].mean() - cate_estimates[old].mean():.2f}")

## Summary

This notebook demonstrated four advanced causal inference methods:

1. **RDD** - Exploits sharp cutoffs in treatment assignment
2. **Synthetic Control** - Creates artificial control units for comparative case studies
3. **Mediation Analysis** - Decomposes effects into direct and indirect pathways
4. **CATE** - Estimates heterogeneous treatment effects across subgroups

### When to Use Each Method:

- **RDD**: When treatment is assigned based on a threshold (test scores, age cutoffs, etc.)
- **Synthetic Control**: For case studies with one treated unit and multiple controls
- **Mediation**: To understand mechanisms and pathways of treatment effects
- **CATE**: When treatment effects vary across individuals or subgroups

### Key Insights:

- **RDD** requires continuity assumption and sufficient data near the cutoff
- **Synthetic Control** works best with good pre-treatment fit
- **Mediation** assumes no unmeasured confounding of mediator-outcome relationship
- **CATE** helps identify who benefits most from treatment

### Next Steps:

- Explore causal graph construction and visualization
- Learn about sensitivity analysis and robustness checks
- Apply these methods to real-world datasets